In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv("C:/Projects/Retail Sales Prediction & Business Performance Analytics/data/merged_data.csv", parse_dates=['Date'])

df.head()

C:\Users\afnan\AppData\Local\Temp\ipykernel_1636\66544211.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Projects/Retail Sales Forecasting & Business Dashboard/data/merged_data.csv", parse_dates=['Date'])


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No_Promo
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,No_Promo
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,No_Promo


Date Features

In [51]:
df['Year'] = df['Date'].dt.year

df['Month'] = df['Date'].dt.month

df['Quarter'] = df['Date'].dt.quarter

df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)

df['Day'] = df['Date'].dt.day

df['DayOfYear'] = df['Date'].dt.dayofyear

df['Weekday'] = df['Date'].dt.day_name()

df['MonthName'] = df['Date'].dt.month_name()

df['IsHoliday'] = (df['StateHoliday'] != '0').astype(int)

Weekend Feature

In [52]:
df['IsWeekend'] = (df['DayOfWeek'].isin([6,7]).astype(int))

Month Start & End

In [53]:
df['IsMonthStart'] = (df['Date'].dt.is_month_start.astype(int))

df['IsMonthEnd'] = (df['Date'].dt.is_month_end.astype(int))

Competition Duration

In [54]:
competition_year = (
    df['CompetitionOpenSinceYear']
        .replace(0, np.nan)
)

competition_month = (
    df['CompetitionOpenSinceMonth']
        .replace(0, 1)
)

competition_date = pd.to_datetime(
    dict(
        year=competition_year,
        month=competition_month,
        day=1
    ),
    errors='coerce'
)

df['CompetitionMonthsOpen'] = (
    (
        df['Date'] - competition_date
    ).dt.days // 30
)

df['CompetitionMonthsOpen'] = (
    df['CompetitionMonthsOpen']
        .fillna(0)
        .clip(lower=0)
)

Promo Duration

In [55]:
promo_year = (
    df['Promo2SinceYear']
        .replace(0, np.nan)
)

promo_week = (
    df['Promo2SinceWeek']
        .replace(0, 1)
)

promo_date = pd.to_datetime(
    promo_year.astype("Int64").astype(str)
    + "-W"
    + promo_week.astype(int).astype(str).str.zfill(2)
    + "-1",
    format="%Y-W%W-%w",
    errors="coerce"
)

df['PromoMonths'] = (
    (
        df['Date'] - promo_date
    ).dt.days // 30
)

df['PromoMonths'] = (
    df['PromoMonths']
        .fillna(0)
        .clip(lower=0)
)

Sales Per Customer

In [56]:
df['SalesPerCustomer'] = (
    df['Sales'] /
    df['Customers']
)

df['SalesPerCustomer'] = (
    df['SalesPerCustomer']
      .replace(np.inf,0)
      .fillna(0)
)

Competition Nearby

In [57]:
df['CompetitionNearby'] = (
    df['CompetitionDistance'] < 1000
).astype(int)

Store Age

In [58]:
first_sale = df['Date'].min()

df['StoreAgeDays'] = (
    df['Date'] -
    first_sale
).dt.days

Lag Features

In [59]:
df = df.sort_values(
    ['Store','Date']
)

In [60]:
#Previous day sales
df['Lag_1'] = (
    df.groupby('Store')['Sales']
      .shift(1)
)

In [61]:
#Previous week
df['Lag_7'] = (
    df.groupby('Store')['Sales']
      .shift(7)
)

In [62]:
#Previous month
df['Lag_30'] = (
    df.groupby('Store')['Sales']
      .shift(30)
)

In [63]:
df[['Lag_1','Lag_7','Lag_30']] = (
    df[['Lag_1','Lag_7','Lag_30']]
      .fillna(0)
)

Rolling Mean

In [64]:
df['Rolling7Mean'] = (
    df.groupby('Store')['Sales']
      .transform(
          lambda x:
          x.shift(1)
           .rolling(7)
           .mean()
      )
)

In [65]:
#Rolling 30
df['Rolling30Mean'] = (
    df.groupby('Store')['Sales']
      .transform(
          lambda x:
          x.shift(1)
           .rolling(30)
           .mean()
      )
)

In [66]:
df[['Rolling7Mean','Rolling30Mean']] = (
    df[['Rolling7Mean','Rolling30Mean']]
      .fillna(0)
)

Rolling Std

In [67]:
df['Rolling7Std'] = (
    df.groupby('Store')['Sales']
      .transform(
          lambda x:
          x.shift(1)
           .rolling(7)
           .std()
      )
)

df['Rolling7Std'] = (
    df['Rolling7Std']
      .fillna(0)
)

In [68]:
print(df.shape)

df.head()

(1017209, 41)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Quarter,WeekOfYear,Day,DayOfYear,Weekday,MonthName,IsHoliday,IsWeekend,IsMonthStart,IsMonthEnd,CompetitionMonthsOpen,PromoMonths,SalesPerCustomer,CompetitionNearby,StoreAgeDays,Lag_1,Lag_7,Lag_30,Rolling7Mean,Rolling30Mean,Rolling7Std
1016095,1,2,2013-01-01,0,0,0,0,a,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No_Promo,2013,1,1,1,1,1,Tuesday,January,1,0,1,0,52.0,0.0,0.000000,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1014980,1,3,2013-01-02,5530,668,1,0,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No_Promo,2013,1,1,1,2,2,Wednesday,January,0,0,0,0,52.0,0.0,8.278443,0,1,0.0,0.0,0.0,0.0,0.0,0.0
1013865,1,4,2013-01-03,4327,578,1,0,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No_Promo,2013,1,1,1,3,3,Thursday,January,0,0,0,0,52.0,0.0,7.486159,0,2,5530.0,0.0,0.0,0.0,0.0,0.0
1012750,1,5,2013-01-04,4486,619,1,0,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No_Promo,2013,1,1,1,4,4,Friday,January,0,0,0,0,52.0,0.0,7.247173,0,3,4327.0,0.0,0.0,0.0,0.0,0.0
1011635,1,6,2013-01-05,4997,635,1,0,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No_Promo,2013,1,1,1,5,5,Saturday,January,0,1,0,0,52.0,0.0,7.869291,0,4,4486.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df.to_csv("C:/Projects/Retail Sales Prediction & Business Performance Analytics/data/featured_data.csv", index=False)